# MLlib

Spark biedt ook een framework aan voor MachineLearning modellen te trainen op gedistribueerde datasets.
Dit framework is MLlib of ook wel sparkML genoemd.
De code om te werken met deze package is sterk gelijkaardig aan sklearn.
De API en een uitgebreide documentatie met voorbeeldcode kan je [hier](https://spark.apache.org/docs/latest/ml-guide.html) vinden.

Deze package bied de volgende tools aan
* ML-technieken: classificatie, regressie, clustering, ...
* Features: Extracting en transforming van features, PCA, ...
* Pipelines: Maak, train, optimaliseer en evalueer pipelines
* Persistentie: Bewaar en laden van algoritmes/modellen
* Databeheer: Algebra tools, statistieken, null-waarden, ...

Let op dat er twee API's aangeboden worden, 1 gebaseerd op RDD's en 1 op DataFrames.
De API gebaseerd op RDD's is ouder en minder flexibel dan de API gebruik makend van DataFrames.
Momenteel werken ze allebei maar in de toekomst zou de RDD gebaseerde kunnen verdwijnen.

## Utilities

### Varianten voor numpy-arrays

Voor feature sets en volledige matrices van datasets aan te maken kan je gebruik maken van de Vector en Matrix klassen.
Deze beschikken over een Dense variant waar je elk element moet ingeven of een Sparse Variant waar cellen, elementen leeg kan laten.
Dit ziet er als volgt uit:

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors
from pyspark.ml.linalg import Matrices

spark = SparkSession.builder \
    .appName("SparkMLlibDemo") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.301") \
    .config("spark.hadoop.fs.s3a.access.key", "bigdata") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
    # voor studenten die problemen hebben met het connecteren van eu-west-1
   # .config("spark.hadoop.fs.s3a.endpoint.region", "eu-west-1") \

sc = spark.sparkContext

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bb93def6-7613-45d3-bf92-02b9bb9e0be7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.301 in central
:: resolution report :: resolve 178ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.301 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 by [com.amazonaws#aws-java-sdk-bundle;1.12.301] in [default]
	---------------------------------------------------------------------
	|      

In [4]:
data = Vectors.dense([4.0, 5.0, 6.0, 3.0])
print(data)

data = Vectors.sparse(10, [(2, 3.0), (7, 8.0)])
print(data)

data = Matrices.dense(4, 4, range(16))
print(data)

[4.0,5.0,6.0,3.0]
(10,[2,7],[3.0,8.0])
DenseMatrix([[ 0.,  4.,  8., 12.],
             [ 1.,  5.,  9., 13.],
             [ 2.,  6., 10., 14.],
             [ 3.,  7., 11., 15.]])


Het is belangrijk om te weten dat dit locale datastructuren (wrapper rond numpy array) zijn en geen gedistribueerde objecten.

### Statistieken

Voor er kan gewerkt worden met statistieken moeten we (net zoals bij pandas) eerst een dataset hebben.
Hieronder maken we een random dataframe aan van 50 rijen en 4 kolommen.

In [7]:
from pyspark.sql.functions import rand
n = 50
d = 4

# Start from range(n) and add d random columns
df = spark.range(0, n)
for i in range(d):
    df = df.withColumn(f"c{i}", rand(seed=42 + i))

df.show(5)

df.summary().show()

+---+-------------------+------------------+------------------+-------------------+
| id|                 c0|                c1|                c2|                 c3|
+---+-------------------+------------------+------------------+-------------------+
|  0|  0.619189370225301|0.8017532427858894|0.7985334687187781| 0.7971301351894658|
|  1| 0.5096018842446481|0.6565552949992319|0.8679617292463396| 0.2422600602366628|
|  2| 0.8325259388871524|0.2515595782593636|0.5068407664758031| 0.7097364852287148|
|  3|0.26322809041172357|0.2073428376111074|0.7946993123364816| 0.4509378549789149|
|  4| 0.6702867696264135|0.6392921379278927|0.7463770360606422|0.30357970527906064|
+---+-------------------+------------------+------------------+-------------------+
only showing top 5 rows



26/03/27 13:23:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 2:=======================================>                   (2 + 1) / 3]

+-------+------------------+-------------------+--------------------+--------------------+--------------------+
|summary|                id|                 c0|                  c1|                  c2|                  c3|
+-------+------------------+-------------------+--------------------+--------------------+--------------------+
|  count|                50|                 50|                  50|                  50|                  50|
|   mean|              24.5| 0.5929594659966628|  0.5392205982358973|  0.4722234723982676| 0.46231880514293267|
| stddev|14.577379737113251| 0.2733627501627969|  0.2796018178158059|  0.2820390630219527| 0.28878456904991756|
|    min|                 0|0.06993233728279169|0.057394743608470966|0.003143456948739...|0.003143456948739...|
|    25%|                12| 0.3724113862805264|  0.2741303337234684| 0.24589255141259847| 0.17883447359405913|
|    50%|                24| 0.6520025939987977|  0.5844552193973006| 0.44821411386180077|  0.4473172424

**Correlation matrix**

Buiten de statistieken die berekend kunnen worden door de summary() functie kan ook de correlatiematrix belangrijk zijn.
Deze matrix maakt het mogelijk om het verband tussen de verscheidene features te bestuderen.
Deze matrix kan als volgt berekend worden voor een gedistribueerd dataframe.

In [13]:
# Eerst moeten we een kolom maken dat alle numerieke waarden groepeert in een vector
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=df.columns, outputCol='vector') # heel vaak moet je het volgende doen: inputkolom(men), outputkolom
# heel vaak outputkolom een nieuwe kolom, vermijd overschrijven om fouten te vermijden, debuggen te vereenvoudigen, ...
df_vector = assembler.transform(df)
df_vector.show(5)

# op basis van dit df kan de correlatie matrix berekend worden
from pyspark.ml.stat import Correlation

df_corr = Correlation.corr(df_vector, 'vector')
df_corr.collect()[0][0].values

+---+-------------------+------------------+------------------+-------------------+--------------------+
| id|                 c0|                c1|                c2|                 c3|              vector|
+---+-------------------+------------------+------------------+-------------------+--------------------+
|  0|  0.619189370225301|0.8017532427858894|0.7985334687187781| 0.7971301351894658|[0.0,0.6191893702...|
|  1| 0.5096018842446481|0.6565552949992319|0.8679617292463396| 0.2422600602366628|[1.0,0.5096018842...|
|  2| 0.8325259388871524|0.2515595782593636|0.5068407664758031| 0.7097364852287148|[2.0,0.8325259388...|
|  3|0.26322809041172357|0.2073428376111074|0.7946993123364816| 0.4509378549789149|[3.0,0.2632280904...|
|  4| 0.6702867696264135|0.6392921379278927|0.7463770360606422|0.30357970527906064|[4.0,0.6702867696...|
+---+-------------------+------------------+------------------+-------------------+--------------------+
only showing top 5 rows



array([ 1.        , -0.10586805, -0.20483437, -0.36980272,  0.17874643,
       -0.10586805,  1.        , -0.25576945, -0.06443734,  0.18283811,
       -0.20483437, -0.25576945,  1.        , -0.12063474, -0.03252672,
       -0.36980272, -0.06443734, -0.12063474,  1.        , -0.24318595,
        0.17874643,  0.18283811, -0.03252672, -0.24318595,  1.        ])

**Onafhankelijksheidtest**

Naast de correlatiematrix kan het ook belangrijk zijn om de onafhankelijkheid te testen tussen elke feature en een label.
Dit kan uitgevoerd worden door een zogenaamde ChiSquareTest.
Deze krijgt als input een dataframe, de naam van de kolom met de features (als vectors) en de naam van een kolom met de labels.
We kunnen deze test uitvoeren als volgt:

In [17]:
# voeg een label-kolom toe aan df_vector
from pyspark.sql.functions import rand, when

df_ind = df_vector.withColumn('label', when(rand() > 0.5, 1).otherwise(0))
df_ind.show(5)

# chi-square test
from pyspark.ml.stat import ChiSquareTest

ChiSquareTest.test(df_ind, 'vector', 'label', flatten=True).show()

+---+-------------------+------------------+------------------+-------------------+--------------------+-----+
| id|                 c0|                c1|                c2|                 c3|              vector|label|
+---+-------------------+------------------+------------------+-------------------+--------------------+-----+
|  0|  0.619189370225301|0.8017532427858894|0.7985334687187781| 0.7971301351894658|[0.0,0.6191893702...|    0|
|  1| 0.5096018842446481|0.6565552949992319|0.8679617292463396| 0.2422600602366628|[1.0,0.5096018842...|    0|
|  2| 0.8325259388871524|0.2515595782593636|0.5068407664758031| 0.7097364852287148|[2.0,0.8325259388...|    1|
|  3|0.26322809041172357|0.2073428376111074|0.7946993123364816| 0.4509378549789149|[3.0,0.2632280904...|    0|
|  4| 0.6702867696264135|0.6392921379278927|0.7463770360606422|0.30357970527906064|[4.0,0.6702867696...|    1|
+---+-------------------+------------------+------------------+-------------------+--------------------+-----+
o

De Chi-square test wordt op de volgende manieren gebruikt in Machine Learning:
* Feature selectie: Identificeer welke features een significante relatie hebben met de targetvariabele.
* Correlatie tussen categorische variabelen: Evalueer de afhankelijkheid tussen twee categorische variabelen.

De Chi-square test vergelijkt de verwachte en werkelijke frequenties van waarden in de categorieën van een feature ten opzichte van de targetvariabele. Het doel is om te bepalen of de verschillen tussen deze frequenties statistisch significant zijn of niet. Een hoge Chi-square score betekent dat er een grote afhankelijkheid is tussen de feature en de target, wat suggereert dat de feature nuttig is voor voorspellingen.
Als de p-waarde kleiner is dan een vooraf bepaald significantieniveau (bijvoorbeeld 0,05), verwerp je de nullhypothese (dat er geen relatie is), wat betekent dat de feature relevant is.

**Summarizer**

Andere statistieken per kolom kunnen berekend worden door gebruik te maken van de Summarizer klasse:

In [21]:
from pyspark.ml.stat import Summarizer

summarizer = Summarizer.metrics('mean', 'min', 'max')
df_vector.select(summarizer.summary(df_ind.vector)).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|aggregate_metrics(vector, 1.0)                                                                                                                                                                                                                                       |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{[24.5,0.5929594659966628,0.5392205982358973,0.4722234723982675,0.46231880514293267], [0.0,0.06993233728279169,0.057394743608470966,0.0031434569487398534,0.0031434569487398534], [49.0,0.9991441647585968,0.96

Het gebruik maken van de Summarizer maakt het dus mogelijk om rechtstreeks op de feature vectors te werken zonder ze eerst terug te moeten splitsen.

### Pipelines

Pipelines binnen Spark zijn een groep van high-level API's steunend op Dataframes om ML-pipelines aan te maken, optimaliseren en trainen.
De belangrijkste concepten binnen de Pipelines van Spark zijn:
* Dataframe: concept van de dataset
* Transformer: Zet een dataframe om in een ander dataframe
* Estimator: Zet een dataframe om in een model/transformer
* Pipeline: een ketting van transformers en estimators om een flow vast te leggen
* Parameter: API voor parameters van transformers en estimators aan te passen

Gebruik nu onderstaande mini-dataset waar we op basis van een tekstkolom met logistische regressie een bepaald label proberen te voorspellen.
Maak hiervoor een Pipeline uit die bestaat uit de volgende stappen:
* Tokenizer om de tekstkolom te splitsen in de overeenkomstige woorden
* HashingTf om de term frequency van de woorden te bepalen en het om te zetten naar een feature vector
* LogisticRegression Estimator om de voorspelling te doen.

Train daarna deze pipeline en maak de voorspellingen voor de traningsdata.
Hoe accuraat is dit model?

In [26]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import HashingTF, Tokenizer

# Prepare training documents from a list of (id, text, label) tuples.
training = spark.createDataFrame([
    (0, "a b c d e spark", 1.0),
    (1, "b d", 0.0),
    (2, "spark f g h", 1.0),
    (3, "hadoop mapreduce", 0.0)
], ["id", "text", "label"])

training.show()

tokenizer = Tokenizer(inputCol='text', outputCol='tokens') # tekst in woordjes verdelen
hashingTf = HashingTF(inputCol='tokens', outputCol='features') # hoeveel keer komt elk woordje voor
lr = LogisticRegression(maxIter=10, regParam=0.01) # features is de default input van een ml-techniek, label is de default label kolom

pipeline = Pipeline(stages = [tokenizer, hashingTf, lr])

model = pipeline.fit(training)
predictions = model.transform(training)
predictions.show(truncate=False)

+---+----------------+-----+
| id|            text|label|
+---+----------------+-----+
|  0| a b c d e spark|  1.0|
|  1|             b d|  0.0|
|  2|     spark f g h|  1.0|
|  3|hadoop mapreduce|  0.0|
+---+----------------+-----+

+---+----------------+-----+----------------------+----------------------------------------------------------------------------+----------------------------------------+-----------------------------------------+----------+
|id |text            |label|tokens                |features                                                                    |rawPrediction                           |probability                              |prediction|
+---+----------------+-----+----------------------+----------------------------------------------------------------------------+----------------------------------------+-----------------------------------------+----------+
|0  |a b c d e spark |1.0  |[a, b, c, d, e, spark]|(262144,[74920,89530,107107,148981,167694,17355

### Evalueren van een model

In de pyspark.ml package zitten er ook functionaliteiten voor deze modellen te evalueren.
Meer informatie hierover vind je [hier](https://spark.apache.org/docs/2.2.0/mllib-evaluation-metrics.html).

In [27]:
# evalueren van het model
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol='prediction', rawPredictionCol='label')
evaluator.evaluate(predictions)

1.0

### Data sources

Door gebruik te maken van de sparkContext kunnen een reeks standaard databronnen ingelezen worden om datasets uit op te bouwen (Csv, Json, ...).
Daarnaast is het ook mogelijk om een folder met een reeks beelden te gebruiken als dataset om zo een model voor image classification te trainen.
Download nu [deze](https://www.kaggle.com/returnofsputnik/chihuahua-or-muffin) dataset en upload ze naar een folder op het hadoop filesysteem.

In [28]:
import kagglehub
from minio import Minio
import os

path = kagglehub.dataset_download("returnofsputnik/chihuahua-or-muffin")

client = Minio(
    "minio1:9000",
    access_key="bigdata",
    secret_key="bigdata123",
    secure=False
)

bucket_name = "03-mllib"

# Make bucket if it doesn't exist
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)

# Upload all images in the folder
for root, dirs, files in os.walk(path):
    for file in files:
        file_path = os.path.join(root, file)
        object_name = file  # You can include subfolders by using relative path if you want
        client.fput_object(bucket_name, f"chimuffin/{object_name}", file_path)
        print(f"Uploaded: {object_name}")

Uploaded: muffin-7.jpeg
Uploaded: muffin-2.jpeg
Uploaded: chihuahua-1.jpg
Uploaded: muffin-3.jpeg
Uploaded: muffin-1.jpeg
Uploaded: chihuahua-6.jpg
Uploaded: chihuahua-8.jpg
Uploaded: chihuahua-7.jpg
Uploaded: chihuahua-5.jpg
Uploaded: chihuahua-3.jpg
Uploaded: muffin-5.jpeg
Uploaded: muffin-8.jpeg
Uploaded: muffin-4.jpeg
Uploaded: muffin-6.jpeg
Uploaded: chihuahua-2.jpg


De geuploade images kunnen nu ingelezen worden als volgt:

In [41]:
df = spark.read.format('image').option('dropInvalid', True).load('s3a://03-mllib/chimuffin')
#dropInvalid: negeer bestanden die geen images zijn

df.printSchema()

root
 |-- image: struct (nullable = true)
 |    |-- origin: string (nullable = true)
 |    |-- height: integer (nullable = true)
 |    |-- width: integer (nullable = true)
 |    |-- nChannels: integer (nullable = true)
 |    |-- mode: integer (nullable = true)
 |    |-- data: binary (nullable = true)



In [42]:
df = df.select('image.origin', 'image.width', 'image.height', 'image.data')
df.show()

+--------------------+-----+------+--------------------+
|              origin|width|height|                data|
+--------------------+-----+------+--------------------+
|s3a://03-mllib/ch...|  172|   170|[79 69 6A 82 72 7...|
|s3a://03-mllib/ch...|  171|   172|[FE FE FE FE FE F...|
|s3a://03-mllib/ch...|  171|   171|[0F 0F 15 0E 0E 1...|
|s3a://03-mllib/ch...|  172|   172|[00 44 9F 0E 5F A...|
|s3a://03-mllib/ch...|  172|   169|[D6 C1 B9 D7 C5 B...|
|s3a://03-mllib/ch...|  168|   172|[A8 A1 B0 94 8C 9...|
|s3a://03-mllib/ch...|  168|   169|[C3 C5 C5 D3 C3 D...|
|s3a://03-mllib/ch...|  171|   169|[80 6C B7 71 5B B...|
|s3a://03-mllib/ch...|  168|   171|[1F 2E 48 20 2F 4...|
|s3a://03-mllib/ch...|  171|   169|[CC CF DD CD D0 D...|
|s3a://03-mllib/ch...|  171|   170|[D3 E2 E5 D0 DF E...|
|s3a://03-mllib/ch...|  171|   170|[57 55 6B 59 56 6...|
|s3a://03-mllib/ch...|  168|   170|[17 1A 28 00 00 0...|
|s3a://03-mllib/ch...|  171|   172|[06 04 68 02 05 6...|
|s3a://03-mllib/ch...|  171|   

Merk op dat het werken met images niet zo eenvoudig is.
Hiervoor wordt binnen pyspark typisch gebruik gemaakt van de [sparkdl](https://smurching.github.io/spark-deep-learning/site/api/python/sparkdl.html) package.
Hierbij staat de dl voor deep learning.
Aangezien dit ons momenteel te ver leidt ga ik dit niet verder toelichten.

Een andere aparte databron die eenvoudig ingelezen kan worden is het formaat "libsvm".
Een bestand van dit formaat wordt ingelezen als een dataframe met twee kolommen: een label en een kolom met de feature-vectors.
De code om dergelijk bestand in te laden is:

In [ ]:
#df = spark.read.format('libsvm').option('dropInvalid', True).load('s3a://03-mllib/chimuffin')
# geeft 2 kolommen
    # feature kolom (dat eigenlijk al een vector) -> geen vectorassembler
    # target kolom 

## Voorbeeld

Train en evalueer een machine learning model dat beeldgegevens kan classificeren, gebaseerd op eigenschappen zoals hoogte, breedte en aantal kanalen. Omdat MLlib geen directe ondersteuning biedt voor beelddata, zullen we enkele numerieke eigenschappen van de afbeeldingen gebruiken en pipelines opzetten voor preprocessing en training.
Hierbij ga je verder werken op het dataframe van de chichuahua of muffin dataset van hierboven.

Voer de volgende stappen uit:
* Data decoderen en omzetten naar een dataframe dat door ML-kan gebruikt worden.
    * Schrijf een udf om de image.data kolom om te zetten naar een vector. Gebruik hiervoor een udf dat dit eerst omzet naar een numpy array waarna het omgezet wordt naar een PIL-Image (een python package). Dit wordt dan geresized naar een 32x32 figuur. Ten slotte wordt dit omgezet naar een 1D-array 
    * Schrijf een udf om de filename in de image.origin kolom om te zetten naar 1 van de labels: 'muffin' of 'chihuahua'
* Na het uitvoeren van de udf's zou je een dataframe moeten hebben met minstens 2 kolommen: het label als string kolom, de data als een vector kolom
* Splits de data in een trainings en testset
* Voer de volgende preprocessing stappen uit
    * Zet de tekstuele labels om naar integers
    * Schaal de vector-kolom door elke waarde te delen door 255 zodat de pixel-waarden in de range 0-255 vallen
    * Zorg ervoor dat dit in een pipeline zit en fit de preprocessor al eens. Print het schema en een aantal rijen uit om de output te verifieren
* Voer data modelling uit
    * Zorg voor dimensionality reduction aan de hand van PCA
    * Gebruik logistische regressie om het label te voorspellen
* Evalueer het model door een confusion matrix te berekenen

In [47]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, StringType
from pyspark.ml.feature import StringIndexer, MinMaxScaler, PCA
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.functions import array_to_vector
from PIL import Image
import io
import numpy as np

def get_label(path): #dit is niet gekend in de containers
    if 'chihuahua' in path:
        return 'chihuahua'
    else:
        return 'muffin'

get_label_udf = F.udf(get_label, StringType()) # udf = user defined function -> hierdoor wordt de functie gekend in de containers
df_data = df.withColumn('label', get_label_udf(df.origin))
df_data.show(5)

def resize_image(pixel_data, img_width, img_height, target_size=(32,32)):
    # 1d vector van spark -> numpy array en 3d (width, height, rgb)
    image_array = np.array(pixel_data, dtype=np.uint8).reshape((img_width, img_height, 3))
    # numpy array -> pil image
    image = Image.fromarray(image_array, mode='RGB')
    # pil -> resize
    image = image.resize(target_size)
    # image -> list , flatten 3d -> 1d
    image = np.array(image, dtype=float).flatten().tolist()
    return image

resize_image_udf = F.udf(resize_image, ArrayType(DoubleType()))
df_data = df_data.withColumn('resized', array_to_vector(resize_image_udf(df_data.data, df_data.width, df_data.height)))
df_data.show(5)
df_data.printSchema()

+--------------------+-----+------+--------------------+---------+
|              origin|width|height|                data|    label|
+--------------------+-----+------+--------------------+---------+
|s3a://03-mllib/ch...|  172|   170|[79 69 6A 82 72 7...|   muffin|
|s3a://03-mllib/ch...|  171|   172|[FE FE FE FE FE F...|   muffin|
|s3a://03-mllib/ch...|  171|   171|[0F 0F 15 0E 0E 1...|   muffin|
|s3a://03-mllib/ch...|  172|   172|[00 44 9F 0E 5F A...|   muffin|
|s3a://03-mllib/ch...|  172|   169|[D6 C1 B9 D7 C5 B...|chihuahua|
+--------------------+-----+------+--------------------+---------+
only showing top 5 rows



[Stage 126:>                                                        (0 + 1) / 1]

+--------------------+-----+------+--------------------+---------+--------------------+
|              origin|width|height|                data|    label|             resized|
+--------------------+-----+------+--------------------+---------+--------------------+
|s3a://03-mllib/ch...|  172|   170|[79 69 6A 82 72 7...|   muffin|[127.0,95.0,91.0,...|
|s3a://03-mllib/ch...|  171|   172|[FE FE FE FE FE F...|   muffin|[255.0,255.0,255....|
|s3a://03-mllib/ch...|  171|   171|[0F 0F 15 0E 0E 1...|   muffin|[27.0,28.0,34.0,2...|
|s3a://03-mllib/ch...|  172|   172|[00 44 9F 0E 5F A...|   muffin|[24.0,99.0,155.0,...|
|s3a://03-mllib/ch...|  172|   169|[D6 C1 B9 D7 C5 B...|chihuahua|[166.0,159.0,170....|
+--------------------+-----+------+--------------------+---------+--------------------+
only showing top 5 rows

root
 |-- origin: string (nullable = true)
 |-- width: integer (nullable = true)
 |-- height: integer (nullable = true)
 |-- data: binary (nullable = true)
 |-- label: string (nullabl

In [50]:
label_indexer = StringIndexer(inputCol='label', outputCol='label_index')
scaler = MinMaxScaler(inputCol='resized', outputCol='scaled_features', min=0.0, max=255.0)
preprocessor = Pipeline(stages=[label_indexer, scaler])

train_data, test_data = df_data.randomSplit([0.8, 0.2])

preprocessor_model = preprocessor.fit(train_data)
preprocessor_model.transform(test_data).show()

+--------------------+-----+------+--------------------+------+--------------------+-----------+--------------------+
|              origin|width|height|                data| label|             resized|label_index|     scaled_features|
+--------------------+-----+------+--------------------+------+--------------------+-----------+--------------------+
|s3a://03-mllib/ch...|  171|   172|[FE FE FE FE FE F...|muffin|[255.0,255.0,255....|        1.0|[328.989637305699...|
|s3a://03-mllib/ch...|  168|   171|[1F 2E 48 20 2F 4...|muffin|[38.0,54.0,87.0,4...|        1.0|[42.2797927461139...|
|s3a://03-mllib/ch...|  168|   169|[C3 C5 C5 D3 C3 D...|muffin|[203.0,192.0,228....|        1.0|[260.284974093264...|
+--------------------+-----+------+--------------------+------+--------------------+-----------+--------------------+



In [51]:
pca = PCA(k=40, inputCol='scaled_features', outputCol='features')
lr = LogisticRegression(maxIter=10, labelCol='label_index')

pipeline = Pipeline(stages=[preprocessor, pca, lr])
model = pipeline.fit(train_data)
predictions = model.transform(test_data)
predictions.show()

26/03/27 15:38:27 WARN DAGScheduler: Broadcasting large task binary with size 1115.6 KiB
26/03/27 15:38:27 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:27 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WARN DAGScheduler: Broadcasting large task binary with size 1116.4 KiB
26/03/27 15:38:28 WAR

+--------------------+-----+------+--------------------+------+--------------------+-----------+--------------------+--------------------+--------------------+-----------+----------+
|              origin|width|height|                data| label|             resized|label_index|     scaled_features|            features|       rawPrediction|probability|prediction|
+--------------------+-----+------+--------------------+------+--------------------+-----------+--------------------+--------------------+--------------------+-----------+----------+
|s3a://03-mllib/ch...|  171|   172|[FE FE FE FE FE F...|muffin|[255.0,255.0,255....|        1.0|[328.989637305699...|[-7433.8542762457...|[-7.6509773316050...|  [0.0,1.0]|       1.0|
|s3a://03-mllib/ch...|  168|   171|[1F 2E 48 20 2F 4...|muffin|[38.0,54.0,87.0,4...|        1.0|[42.2797927461139...|[-5159.5777192995...|[-4.2203047219946...|  [0.0,1.0]|       1.0|
|s3a://03-mllib/ch...|  168|   169|[C3 C5 C5 D3 C3 D...|muffin|[203.0,192.0,228....| 

In [55]:
from pyspark.sql import functions as F

confusion_df = (
    predictions
    .groupBy("label_index")
    .pivot("prediction")
    .count()
    .fillna(0)
    .orderBy("label_index")
)

confusion_df.show()

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol='label_index', predictionCol='prediction', metricName='accuracy')

evaluator.evaluate(predictions)

26/03/27 15:42:48 WARN DAGScheduler: Broadcasting large task binary with size 1102.8 KiB
26/03/27 15:42:48 WARN DAGScheduler: Broadcasting large task binary with size 1089.1 KiB
26/03/27 15:42:48 WARN DAGScheduler: Broadcasting large task binary with size 1088.8 KiB
26/03/27 15:42:48 WARN DAGScheduler: Broadcasting large task binary with size 1084.0 KiB
26/03/27 15:42:48 WARN DAGScheduler: Broadcasting large task binary with size 1115.0 KiB
26/03/27 15:42:48 WARN DAGScheduler: Broadcasting large task binary with size 1095.0 KiB
26/03/27 15:42:49 WARN DAGScheduler: Broadcasting large task binary with size 1099.3 KiB


+-----------+---+---+
|label_index|0.0|1.0|
+-----------+---+---+
|        1.0|  1|  2|
+-----------+---+---+



26/03/27 15:42:49 WARN DAGScheduler: Broadcasting large task binary with size 1117.8 KiB
                                                                                

0.6666666666666666

In [56]:
model.write().overwrite().save('s3a://03-mllib/model')

In [58]:
from pyspark.ml import PipelineModel

model_loaded = PipelineModel.load('s3a://03-mllib/model')

## Oefening - classificatie

Voer de volgende stappen uit, volg de best-practices van het data science vak:
* Genereer een dataset met de make_classification functie van Scikit-Learn (of gebruik randomSplit op een bestaande dataset in PySpark).
* Plaats de dataset op de MinIO cluster
* Converteer de dataset naar een PySpark DataFrame door het te lezen vanaf de MinIO cluster.
* Voer basisverkenning uit: toon het schema van de data, bekijk de eerste rijen, en controleer de kolomtypes.
* Gebruik VectorAssembler om alle features in één vector-kolom (features) te combineren.
* Controleer de structuur van de output.
* Train een Logistic Regression model op de gegenereerde dummy data.
* Gebruik een training/test split van 80/20.
* Evalueer het model door de nauwkeurigheid (accuracy) te berekenen.

## Oefening - hyperparameter tuning

Voer de volgende stappen uit, volg de best-practices van het data science vak:
* Genereer dummy data (of gebruik bestaande data) om een regressieprobleem op te lossen. (make_regression)
* Bouw een pipeline die de data voorbereidt door schaling uit te voeren, een RandomForest-model traint, en de beste hyperparameters selecteert. (Tip: ParamGridBuilder)
* Voer hyperparameter tuning uit en evalueer de prestaties van het beste model.